# ETAPA 1: Análise exploratoria do  dataset NSL-KDD 
Objetivo: caracterizar a  base e indentificar problemas antes da modelagem.

In [31]:
import pandas as pd
import matplotlib.pyplot as plt

## Plotagem do dataset como tabela

### Codigo para coletar os nomes dos Atributos

Os arquivos .txt não têm cabeçalho. Os nomes das colunas foram lidos do cabeçalho do arquivo .arff, nas linhas que começam com @attribute.

In [2]:
def ler_nomes_arff(caminho):
                nomes = []
                with open(caminho) as arquivo:
                    for linha in arquivo:
                        if linha.startswith("@attribute"):
                            partes = linha.split()
                            nome = partes[1].strip("'")
                            nomes.append(nome)
                        if linha.startswith("@data"):
                            break
                return nomes
nomes_arff = ler_nomes_arff("BANCO_DE_DADOS/Treino.arff")
print(len(nomes_arff))
print(nomes_arff)

42
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class']


### Codigo para a ploatagem da tabela e suas 5 primeiras tuplas

In [3]:
colunas = nomes_arff[:-1] + ["attack_type", "difficulty"]
print(len(colunas))

treino = pd.read_csv("BANCO_DE_DADOS/TREINO.txt", header=None, names = colunas)
teste = pd.read_csv( "BANCO_DE_DADOS/TESTE.txt",   header=None, names = colunas)

print(treino.shape)
print(teste.shape)
treino.head()

43
(125973, 43)
(22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## 2.1 Caracterização da base

### Teste para saber se existe valores ausentes. No pandas, um valor ausente aparece como NaN (“Not a Number”).


In [6]:
print("Ausentes no treino:", treino.isna().sum().sum())
print("Ausentes no teste:", teste.isna().sum().sum())

Ausentes no treino: 0
Ausentes no teste: 0


### Teste para saber se não existe ausência disfarçada, por exemplo, se tem dado faltante escrito como "?".

In [7]:
print("Ausentes no treino:",(treino == "?").sum().sum())
print("Ausentes no teste:",(teste == "?").sum().sum())

Ausentes no treino: 0
Ausentes no teste: 0


Não há valores ausentes explícitos (NaN). Também não há ausências codificadas como ‘?’. Todas as colunas numéricas foram lidas pelo pandas como números, o que não aconteceria se houvesse texto misturado.

### Teste para verificar se a base tem zeros que na verdade significam "sem informação".


Para verificar se os zeros das colunas numéricas são reais ou indicam dado ausente, calculamos a porcentagem de zeros de cada coluna e, entre esses zeros, quantos ocorrem em conexões não estabelecidas (flag S0 ou REJ).

In [13]:
# Seleciona só as colunas numéricas (exclui protocol_type, service, flag e attack_type)
numericas = treino.select_dtypes(include="number").columns

linhas = []                                   # aqui vamos guardar uma linha da tabela por coluna

for col in numericas:                         # percorre cada coluna numérica
    eh_zero = treino[col] == 0                # True/False: a linha tem zero nesta coluna?

    linhas.append({                           # um dicionário = uma linha da tabela final
        "coluna": col,
        "% zeros": eh_zero.mean() * 100,
        # Das linhas com zero, que fração tem flag S0 ou REJ (conexão não estabelecida)?
        "% dos zeros em conexões não estabelecidas":
            treino.loc[eh_zero, "flag"].isin(["S0", "REJ"]).mean() * 100,
            "linhas ≠ 0": (~eh_zero).sum(),     # ~ inverte True/False: conta quem NÃO é zero
    })

tabela_zeros = pd.DataFrame(linhas).set_index("coluna").rename_axis(None).round(1)
tabela_zeros.sort_values("% zeros", ascending=False)

,% zeros,% dos zeros em conexões não estabelecidas,linhas ≠ 0
land,100.0,36.6,25
urgent,100.0,36.6,9
num_shells,100.0,36.6,47
num_outbound_cmds,100.0,36.6,0
is_host_login,100.0,36.6,1
num_failed_logins,99.9,36.6,122
su_attempted,99.9,36.6,80
root_shell,99.9,36.6,169
num_file_creations,99.8,36.7,287
num_access_files,99.7,36.7,371


### **Conclusão da 2.1** 

A base não tem valores ausentes explícitos nem disfarçados.

Os zeros de src_bytes estão concentrados em conexões não estabelecidas: 93,3% deles têm flag S0 ou REJ, contra 36,6% na base inteira. Zero bytes, aqui, significa que nenhum dado foi enviado.

Em dst_bytes, 67,8% dos zeros estão nessas conexões; os demais vêm de conexões em que o destino não devolveu dados.


Nas outras colunas, o zero tem significado pela definição: o evento não ocorreu (contagens), “não” (binárias) ou 0% (taxas). Para colunas quase todas zero, o teste com a flag só reflete a média geral (36,6%) e não é informativo.


#### **Decisão: nenhum tratamento de ausentes é necessário.**

**Observação para a etapa de redundâncias: num_outbound_cmds é 100% zero, e land, urgent, num_shells e is_host_login têm menos de 50 valores diferentes de zero.**

Para identificar os atributos categóricos, verificamos o tipo de cada coluna e quantos valores diferentes ela assume, incluindo colunas numéricas que, na prática, funcionam como categorias (0/1).

## 2.2 Atributos categóricos
Analisamos os valores dos atributos categóricos (`protocol_type`, `service` e `flag`) no treino e no teste, e os atributos numéricos que funcionam como categorias (0/1).

O código abaixo conta quantos valores diferentes cada atributo categórico tem e verifica se algum serviço existe em só uma das bases.

In [48]:
# Quantos valores diferentes cada atributo categórico tem no treino e no teste
for col in ["protocol_type", "service", "flag"]:
    print(f"{col}: {treino[col].nunique()} valores no treino, {teste[col].nunique()} no teste")

# Serviços que existem em só uma das bases (diferença de conjuntos)
print("\nServiços só no treino:", set(treino["service"]) - set(teste["service"]))
print("Serviços só no teste:", set(teste["service"]) - set(treino["service"]))

protocol_type: 3 valores no treino, 3 no teste
service: 70 valores no treino, 64 no teste
flag: 11 valores no treino, 11 no teste

Serviços só no treino: {'urh_i', 'harvest', 'aol', 'red_i', 'http_2784', 'http_8001'}
Serviços só no teste: set()


O código abaixo mostra os valores mais frequentes de cada atributo categórico no treino.

In [49]:
display(treino["protocol_type"].value_counts())
display(treino["flag"].value_counts())
display(treino["service"].value_counts().head(10))   # 10 serviços mais comuns, de 70

protocol_type
tcp     102689
udp      14993
icmp      8291
Name: count, dtype: int64

flag
SF        74945
S0        34851
REJ       11233
RSTR       2421
RSTO       1562
S1          365
SH          271
S2          127
RSTOS0      103
S3           49
OTH          46
Name: count, dtype: int64

service
http        40338
private     21853
domain_u     9043
smtp         7313
ftp_data     6860
eco_i        4586
other        4359
ecr_i        3077
telnet       2353
finger       1767
Name: count, dtype: int64

Alguns atributos numéricos assumem apenas os valores 0 e 1, funcionando como categorias (não/sim). O código abaixo identifica esses atributos.

In [50]:
# Atributos numéricos com exatamente 2 valores (binários: 0 = não, 1 = sim)
print("Binários:", [c for c in numericas if treino[c].nunique() == 2])

# su_attempted deveria ser binário ("tentou su root?"), então deveria ter só 0 e 1
print("su_attempted:", treino["su_attempted"].value_counts().to_dict())

Binários: ['land', 'logged_in', 'root_shell', 'is_host_login', 'is_guest_login']
su_attempted: {0: 125893, 2: 59, 1: 21}


### **Conclusão da 2.2**
- `protocol_type` (3 valores) e `flag` (11) têm os mesmos valores no treino e no teste. A maioria das conexões é `tcp` e termina normalmente (`SF`).
- O `service` tem **alta cardinalidade** (70 valores), o que pesa na escolha da técnica de codificação. 6 serviços aparecem só no treino e nenhum é exclusivo do teste, então o modelo não encontrará serviços desconhecidos na avaliação.
- Os atributos binários (`land`, `logged_in`, `root_shell`, `is_host_login`, `is_guest_login`) já estão codificados como 0/1 e não precisam de conversão.
- O `su_attempted` deveria ser binário, mas tem o valor 2 em 59 linhas (ver seção 5).

## 3. Identificação das classes
A coluna `attack_type` indica se a conexão é normal ou o nome do ataque. Nesta seção,
verificamos quais rótulos existem, agrupamos os ataques em categorias e comparamos
treino e teste.

O código abaixo conta quantas conexões existem de cada rótulo e quantos tipos de ataque diferentes há em cada base. O `normal` é descontado da contagem porque não é um ataque.

In [26]:
# Quantas conexões de cada rótulo existem no treino (do mais comum ao menos comum)
display(treino["attack_type"].value_counts())

# Quantos tipos de ATAQUE existem (o -1 desconta o "normal")
print("Tipos de ataque no treino:", treino["attack_type"].nunique() - 1)
print("Tipos de ataque no teste:", teste["attack_type"].nunique() - 1)

attack_type
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

Tipos de ataque no treino: 22
Tipos de ataque no teste: 37


Como existem muitos tipos de ataque, e vários aparecem pouquíssimas vezes, eles são agrupados nas quatro categorias usadas na literatura:
- **DoS** (negação de serviço): sobrecarregar um alvo até ele parar de responder.
- **Probe** (sondagem): varrer a rede em busca de máquinas, portas abertas e falhas.
- **R2L** (*remote to local*): obter acesso a uma máquina a partir de fora dela.
- **U2R** (*user to root*): um usuário comum obter privilégios de administrador.

O código abaixo cria uma coluna `categoria` com esse agrupamento. No final, verificamos se algum ataque ficou sem categoria.

In [27]:
# Categoria de cada tipo de ataque (mapeamento mais usado em trabalhos com o NSL-KDD)
categorias = {
    "DoS":   ["back", "land", "neptune", "pod", "smurf", "teardrop", "apache2",
              "udpstorm", "processtable", "mailbomb", "worm"],
    "Probe": ["satan", "ipsweep", "nmap", "portsweep", "mscan", "saint"],
    "R2L":   ["guess_passwd", "ftp_write", "imap", "phf", "multihop", "warezmaster",
              "warezclient", "spy", "xlock", "xsnoop", "snmpguess", "snmpgetattack",
              "httptunnel", "sendmail", "named"],
    "U2R":   ["buffer_overflow", "loadmodule", "rootkit", "perl", "sqlattack", "xterm", "ps"],
}

# Inverte o dicionário: de "categoria -> lista de ataques" para "ataque -> categoria"
mapa = {}
for categoria, ataques in categorias.items():
    for ataque in ataques:
        mapa[ataque] = categoria
mapa["normal"] = "Normal"

# .map() troca cada valor de attack_type pela categoria correspondente
treino["categoria"] = treino["attack_type"].map(mapa)
teste["categoria"] = teste["attack_type"].map(mapa)

# Verificação: ataque fora do dicionário vira NaN (os dois valores têm que ser 0)
print("Sem categoria no treino:", treino["categoria"].isna().sum())
print("Sem categoria no teste:", teste["categoria"].isna().sum())

Sem categoria no treino: 0
Sem categoria no teste: 0


O código abaixo conta quantas conexões existem em cada categoria, no treino e no teste.

In [28]:
# Quantidade de conexões por categoria, no treino e no teste
display(treino["categoria"].value_counts())
display(teste["categoria"].value_counts())

categoria
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

categoria
Normal    9711
DoS       7460
R2L       2885
Probe     2421
U2R         67
Name: count, dtype: int64

Se o teste tem mais tipos de ataque que o treino, alguns ataques só existem no teste. O código abaixo identifica quais são e quantas conexões do teste eles representam.

In [29]:
# Conjuntos de tipos de ataque de cada base
ataques_treino = set(treino["attack_type"])
ataques_teste = set(teste["attack_type"])

# Tipos que estão no teste e não estão no treino
so_no_teste = ataques_teste - ataques_treino
print("Quantos tipos só no teste:", len(so_no_teste))
print(so_no_teste)

# Quantas conexões do teste são desses ataques novos (.isin = está no conjunto?)
print("Conexões de ataques novos no teste:", teste["attack_type"].isin(so_no_teste).sum())

Quantos tipos só no teste: 17
{'mailbomb', 'named', 'ps', 'sqlattack', 'httptunnel', 'apache2', 'snmpgetattack', 'worm', 'snmpguess', 'saint', 'udpstorm', 'sendmail', 'mscan', 'xterm', 'xlock', 'processtable', 'xsnoop'}
Conexões de ataques novos no teste: 3750


### **Conclusão da seção 3**

- **Rótulos:** o treino tem 22 tipos de ataque e o teste tem 37, além das conexões normais.

- **Agrupamento:** todos os ataques foram classificados em uma das quatro categorias (a verificação não encontrou nenhum rótulo sem categoria, no treino nem no teste).

- **Distribuição no treino:** Normal (67.343), DoS (45.927), Probe (11.656), R2L (995) e U2R (52). A diferença entre a maior e a menor categoria é enorme: há cerca de 1.300 conexões normais para cada conexão U2R. Esse desbalanceamento é analisado na próxima seção.

- **Distribuição no teste:** Normal (9.711), DoS (7.460), Probe (2.421), R2L (2.885) e U2R (67). As proporções mudam em relação ao treino: o R2L, por exemplo, é bem mais frequente no teste.

- **Ataques novos:** 17 tipos de ataque existem só no teste, somando 3.750 conexões (cerca de 29% dos ataques do teste). 

- **Implicação para a modelagem:** como os ataques novos não podem ser previstos pelo nome, o problema deve ser tratado por **categoria** (Normal, DoS, Probe, R2L, U2R), e não pelo tipo específico de ataque.

- **Implicação para a modelagem:** como 17 tipos de ataque existem apenas no teste, um modelo treinado para prever o nome exato do ataque não teria como acertá-los. Por isso, as formulações usuais com o NSL-KDD são a classificação binária (normal × ataque) ou em cinco classes (Normal, DoS, Probe, R2L e U2R). A formulação adotada será definida pela equipe na etapa de modelagem.

## 5. Inconsistências e redundâncias
**Inconsistências** são dados que se contradizem ou que fogem do que deveria ser possível (por exemplo, um atributo binário com um terceiro valor). **Redundâncias** são dados repetidos: linhas duplicadas ou colunas que não acrescentam informação. As duas afetam o treinamento e precisam ser identificadas antes do pré-processamento.

### 5.1 Linhas duplicadas e contraditórias
O NSL-KDD foi criado para eliminar os registros repetidos do KDD Cup 99. O código abaixo verifica se isso se confirma em dois níveis: linhas 100% iguais e linhas iguais apenas nos 41 atributos (que podem ter rótulos diferentes).

In [41]:
# Os 41 atributos (todas as colunas, menos attack_type e difficulty)
atributos = colunas[:41]

for nome, base in (("treino", treino), ("teste", teste)):
    # duplicated() marca como True cada linha que repete uma linha anterior
    iguais_total = base.duplicated(subset=colunas).sum()        # compara as 43 colunas originais
    iguais_atributos = base.duplicated(subset=atributos).sum()  # compara só os 41 atributos
    print(f"{nome}: linhas 100% iguais = {iguais_total}, iguais nos 41 atributos = {iguais_atributos}")

treino: linhas 100% iguais = 0, iguais nos 41 atributos = 16
teste: linhas 100% iguais = 0, iguais nos 41 atributos = 57


Linhas iguais nos atributos podem ter rótulos diferentes: as mesmas características registradas uma vez como normal e outra como ataque. Essas linhas são **contraditórias**, porque nenhum modelo consegue acertar as duas. O código abaixo identifica essas linhas.

In [43]:
def linhas_contraditorias(base):
    # Copia os 41 atributos e acrescenta uma coluna: a linha é ataque? (True/False)
    tabela = base[atributos].copy()
    tabela["eh_ataque"] = base["attack_type"] != "normal"

    # Agrupa as linhas com atributos idênticos e conta quantos rótulos diferentes cada grupo tem
    rotulos_por_grupo = tabela.groupby(atributos)["eh_ataque"].nunique()

    # Grupos com mais de 1 rótulo = as mesmas características com normal E ataque
    grupos_em_conflito = rotulos_por_grupo[rotulos_por_grupo > 1].index

    # Devolve True para cada linha que pertence a um desses grupos
    return tabela.set_index(atributos).index.isin(grupos_em_conflito)

for nome, base in (("treino", treino), ("teste", teste)):
    print(f"{nome}: {linhas_contraditorias(base).sum()} linhas contraditórias")

# Mostra as linhas contraditórias do treino (algumas colunas, para caber na tela)
colunas_ver = ["protocol_type", "service", "flag", "src_bytes", "count", "attack_type"]
display(treino[linhas_contraditorias(treino)].sort_values(colunas_ver)[colunas_ver])

treino: 14 linhas contraditórias
teste: 108 linhas contraditórias


,protocol_type,service,flag,src_bytes,count,attack_type
16515,icmp,ecr_i,SF,20,1,normal
6922,icmp,ecr_i,SF,20,1,satan
30680,icmp,tim_i,SF,564,1,normal
1143,icmp,tim_i,SF,564,1,pod
67588,tcp,finger,S0,0,1,land
89324,tcp,finger,S0,0,1,land
105684,tcp,finger,S0,0,1,land
121116,tcp,finger,S0,0,1,land
13210,tcp,finger,S0,0,1,normal
37107,tcp,finger,S0,0,1,normal


### 5.2 Valores fora do domínio esperado
O código abaixo verifica se algum atributo assume valores impossíveis pela sua definição: taxas fora do intervalo [0, 1], valores negativos e o atributo `su_attempted`, que deveria ser binário.

In [44]:
# Atributos numéricos (sem as colunas de texto e sem difficulty)
atributos_numericos = [c for c in atributos if c in numericas]

# Taxas: todas as colunas com "rate" no nome devem estar entre 0 e 1
taxas = [c for c in atributos if "rate" in c]
fora_do_intervalo = ((treino[taxas] < 0) | (treino[taxas] > 1)).sum().sum()   # | = "ou"
print("Taxas fora de [0, 1]:", fora_do_intervalo)

# Nenhum atributo numérico deveria ser negativo (são contagens, bytes, durações e taxas)
print("Valores negativos:", (treino[atributos_numericos] < 0).sum().sum())

# su_attempted: "1 se 'su root' foi tentado, 0 caso contrário" -> deveria ter só 0 e 1
print("\nsu_attempted no treino:")
display(treino["su_attempted"].value_counts())
print("su_attempted no teste:")
display(teste["su_attempted"].value_counts())

Taxas fora de [0, 1]: 0
Valores negativos: 0

su_attempted no treino:


su_attempted
0    125893
2        59
1        21
Name: count, dtype: int64

su_attempted no teste:


su_attempted
0    22540
2        2
1        2
Name: count, dtype: int64

### 5.3 Colunas constantes ou quase constantes
Uma coluna com o mesmo valor em todas as linhas não ajuda o modelo a diferenciar nada. O código abaixo procura colunas constantes e colunas em que um único valor ocupa mais de 99,9% das linhas.

In [45]:
# Colunas com um único valor em todas as linhas
constantes = [c for c in atributos_numericos if treino[c].nunique() == 1]
print("Colunas constantes:", constantes)

# Para cada coluna: % de linhas ocupadas pelo valor mais comum
mais_comum = treino[atributos_numericos].apply(
    lambda coluna: coluna.value_counts(normalize=True).iloc[0] * 100
)
display(mais_comum[mais_comum > 99.9].sort_values(ascending=False).round(3))

Colunas constantes: ['num_outbound_cmds']


num_outbound_cmds    100.000
is_host_login         99.999
urgent                99.993
land                  99.980
num_shells            99.963
su_attempted          99.936
num_failed_logins     99.903
dtype: float64

### 5.4 Colunas correlacionadas
Duas colunas muito correlacionadas carregam quase a mesma informação. A **correlação de Pearson** vai de −1 a 1: perto de 1, as duas sobem juntas; perto de −1, uma sobe e a outra desce; perto de 0, não há relação linear. O código abaixo calcula a correlação entre os atributos numéricos e lista os pares com correlação de 0,9 ou mais (em valor absoluto).

In [46]:
# Correlação entre os atributos numéricos (sem as constantes, que não têm correlação definida)
atributos_corr = [c for c in atributos_numericos if c not in constantes]
correlacao = treino[atributos_corr].corr()

# Percorre cada par de atributos uma vez só e guarda os pares com |correlação| >= 0,9
pares = []
for i, a in enumerate(atributos_corr):
    for b in atributos_corr[i + 1:]:          # só os atributos depois de "a" (evita repetir pares)
        r = correlacao.loc[a, b]
        if abs(r) >= 0.9:
            pares.append((a, b, round(r, 3)))

pares_fortes = pd.DataFrame(pares, columns=["atributo A", "atributo B", "correlação"])
print(len(pares_fortes), "pares com correlação >= 0,9")
display(pares_fortes.sort_values("correlação", ascending=False))

13 pares com correlação >= 0,9


,atributo A,atributo B,correlação
0,num_compromised,num_root,0.999
1,serror_rate,srv_serror_rate,0.993
6,rerror_rate,srv_rerror_rate,0.989
5,srv_serror_rate,dst_host_srv_serror_rate,0.986
11,dst_host_serror_rate,dst_host_srv_serror_rate,0.985
3,serror_rate,dst_host_srv_serror_rate,0.981
2,serror_rate,dst_host_serror_rate,0.979
4,srv_serror_rate,dst_host_serror_rate,0.978
10,srv_rerror_rate,dst_host_srv_rerror_rate,0.970
8,rerror_rate,dst_host_srv_rerror_rate,0.964


### **Conclusão da seção 5**

**Inconsistências**
- Não há linhas 100% duplicadas, o que confirma a proposta do NSL-KDD.
- Há 16 linhas no treino (57 no teste) iguais a outra nos 41 atributos. Entre elas, **14 linhas do treino e 108 do teste são contraditórias**: as mesmas características aparecem como normal e como ataque (por exemplo, `normal` × `land`). Nenhum modelo acerta as duas, então elas funcionam como ruído de rótulo. São poucas e não devem alterar os resultados, mas o tratamento (remover ou manter) será discutido na Etapa 2.
- Não há taxas fora de [0, 1] nem valores negativos.
- **`su_attempted` deveria ser binário, mas assume o valor 2** em 59 linhas do treino e 2 do teste. Como a documentação não define o valor 2, a correção mais comum é tratá-lo como 1.

**Redundâncias**
- **`num_outbound_cmds` é constante** (sempre 0) e deve ser removido, pois não carrega informação.
- **`is_host_login` é quase constante**: apenas 1 linha do treino tem valor diferente de 0.
- Há **13 pares de atributos com correlação ≥ 0,9**, que formam três grupos:
  1. **Erro de SYN:** `serror_rate`, `srv_serror_rate`, `dst_host_serror_rate`, `dst_host_srv_serror_rate`;
  2. **Rejeição:** `rerror_rate`, `srv_rerror_rate`, `dst_host_rerror_rate`, `dst_host_srv_rerror_rate`;
  3. **Acesso root:** `num_compromised` e `num_root` (correlação 0,999).

  Em cada grupo, os atributos medem a mesma coisa em janelas ou recortes diferentes, então trazem informação repetida. Essa redundância deve ser considerada na etapa de redução de dimensionalidade.

## 6. Possíveis problemas de vazamento de dados
**Vazamento de dados** (*data leakage*) acontece quando o modelo tem acesso, durante o treinamento, a uma informação que não estaria disponível no uso real, como o próprio rótulo ou dados do conjunto de teste. O resultado é um desempenho otimista que não se repete na prática.

### 6.1 Atributos classificatórios
Das 43 colunas, 41 descrevem a conexão e serão as entradas do modelo. A coluna `attack_type` é a **classe**: indica se a conexão é normal ou o tipo de ataque. A coluna `difficulty` não descreve a conexão nem é a classe; ela registra quantos de 21 classificadores acertaram a classe de cada conexão.

Além da classe, um atributo pode "classificar sem querer" o registro: um identificador único (como ID, IP ou horário), por exemplo, permitiria ao modelo decorar cada linha. O código abaixo verifica se existe alguma coluna com um valor diferente em cada linha.

In [51]:
# Uma coluna identificadora teria um valor diferente para cada linha (nunique == número de linhas)
identificadores = [c for c in colunas if treino[c].nunique() == len(treino)]
print("Colunas identificadoras:", identificadores)

Colunas identificadoras: []


### 6.2 Correlação de cada atributo com a classe
Um atributo com correlação muito alta com a classe (próxima de 1) pode estar revelando a resposta ao modelo. O código abaixo calcula a correlação de Pearson de cada atributo numérico com a classe binária (1 = ataque, 0 = normal). A coluna `difficulty` foi incluída para comparação.

In [52]:
# Classe binária como número: True/False -> 1/0
eh_ataque = treino["attack_type"] != "normal"

# Atributos numéricos não constantes, mais a coluna difficulty
atributos = colunas[:41]
numericos = [c for c in atributos if c in numericas and treino[c].nunique() > 1] + ["difficulty"]

# corrwith calcula a correlação de cada coluna com a classe; abs() ignora o sinal (+ ou -)
corr_classe = treino[numericos].corrwith(eh_ataque.astype(int)).abs().sort_values(ascending=False)
display(corr_classe.head(10).round(3))

same_srv_rate               0.752
dst_host_srv_count          0.723
dst_host_same_srv_rate      0.694
logged_in                   0.690
dst_host_srv_serror_rate    0.655
dst_host_serror_rate        0.652
serror_rate                 0.651
srv_serror_rate             0.648
count                       0.576
difficulty                  0.380
dtype: float64

### 6.3 A coluna `difficulty`
Para calcular `difficulty`, os autores da base precisaram do rótulo verdadeiro, então ela não existiria para uma conexão nova. O código abaixo verifica se ela varia entre as classes e quanto uma regra simples acerta usando apenas essa coluna: para cada nível de dificuldade, a regra prevê a classe mais comum naquele nível.

In [53]:
# Dificuldade média e mediana de cada categoria
ordem = ["Normal", "DoS", "Probe", "R2L", "U2R"]
display(treino.groupby("categoria")["difficulty"].agg(["mean", "median"]).reindex(ordem).round(1))

# Regra simples: para cada valor de difficulty, a fração de ataques é maior que 50%?
# Se for, a regra prevê "ataque" para todas as conexões com esse valor
regra = eh_ataque.groupby(treino["difficulty"]).mean() > 0.5
previsao = treino["difficulty"].map(regra)

print(f"Acerto usando só difficulty: {(previsao == eh_ataque).mean() * 100:.1f}%")
print(f"Acerto chutando sempre 'normal': {(~eh_ataque).mean() * 100:.1f}%")

,mean,median
categoria,,
Normal,20.3,21.0
DoS,19.3,20.0
Probe,16.5,17.0
R2L,9.9,11.0
U2R,4.4,3.0


Acerto usando só difficulty: 74.9%
Acerto chutando sempre 'normal': 53.5%


### 6.4 Linhas repetidas entre treino e teste
O enunciado afirma que treino e teste não têm instâncias duplicadas entre si. O código abaixo verifica isso, comparando as 43 colunas de cada linha do teste com as linhas do treino.

In [54]:
# Transforma cada linha (43 colunas originais) em uma "chave" única, para comparar linhas inteiras
chave_treino = pd.MultiIndex.from_frame(treino[colunas].astype(str))
chave_teste = pd.MultiIndex.from_frame(teste[colunas].astype(str))

# isin: True para cada linha do teste que também existe no treino
repetidas = chave_teste.isin(chave_treino)
print(f"Linhas do teste idênticas a linhas do treino: {repetidas.sum()} ({repetidas.mean() * 100:.1f}% do teste)")

# Quais rótulos têm essas linhas repetidas
display(teste.loc[repetidas, "attack_type"].value_counts().head())


Linhas do teste idênticas a linhas do treino: 610 (2.7% do teste)


attack_type
smurf      203
neptune    155
ipsweep     97
normal      87
pod         21
Name: count, dtype: int64

### 6.5 Cuidados para evitar vazamento nas próximas etapas
- **Rótulos fora dos atributos:** `attack_type`, `categoria` e `difficulty` não podem ser usados como entrada do modelo.
- **Pré-processamento ajustado só no treino:** normalização, codificação, balanceamento e seleção de atributos devem aprender apenas com os dados de treino e depois ser aplicados ao teste. O `Pipeline` do scikit-learn garante isso, inclusive dentro da validação cruzada.
- **Teste usado uma única vez:** escolhas de pré-processamento, hiperparâmetros e modelo devem ser feitas com validação cruzada estratificada no treino. O KDDTest+ é usado apenas na avaliação final.
- **Arquivos auxiliares:** segundo a documentação da base, o `KDDTrain+_20Percent` é um subconjunto do `KDDTrain+`, então não pode ser usado como validação de um modelo treinado no conjunto completo.

## 7. Diagnóstico

| Aspecto | Achado | Ação recomendada |
|---|---|---|
| **Base** | 125.973 conexões no treino e 22.544 no teste; 41 atributos + classe (`attack_type`) + dificuldade | nomes das colunas lidos do `.arff` |
| **Valores ausentes** | nenhum, nem explícito nem disfarçado; os zeros têm significado | nenhuma |
| **Atributos categóricos** | `protocol_type` (3), `flag` (11) e `service` (70, alta cardinalidade); 6 serviços só no treino, nenhum só no teste | escolher a codificação considerando a cardinalidade do `service` |
| **Classes** | 22 tipos de ataque no treino e 37 no teste, agrupados em DoS, Probe, R2L e U2R; 17 tipos só existem no teste | prever o nome exato do ataque é inviável; usar a formulação binária ou em 5 classes, a ser definida na modelagem |
| **Desbalanceamento** | binário equilibrado (53,5% × 46,5%); em 5 classes, 1.295 normais para cada U2R; proporções mudam no teste | F1-macro; validação estratificada; balanceamento só no treino |
| **Inconsistências** | 14 linhas contraditórias no treino; `su_attempted` com valor 2 | avaliar a remoção das contraditórias; tratar `su_attempted = 2` como 1 |
| **Redundâncias** | `num_outbound_cmds` constante; `is_host_login` quase constante; 3 grupos de atributos com correlação ≥ 0,9 | remover a constante; considerar os grupos na redução de dimensionalidade |
| **Vazamento** | `difficulty` deriva do rótulo (acerta 74,9% sozinha); sem identificadores; 610 linhas do teste repetem o treino | remover `difficulty` e os rótulos dos atributos; pré-processamento só no treino; teste usado uma única vez |